# AquaCubeAI — spatial inference and mapping (20 × 20)

Use the supplied spatial checkpoint to create the three-panel mapping figure:
CMEMS reference turbidity, predicted turbidity and the >100 FNU anomaly mask.
The full GeoTIFF is the background; inference uses native 20 × 20 patches at
retained reference locations. This is separate from full-scene device deployment.

No upper clipping or outlier removal is applied to predictions. The target
filter `[0, 140)` applies to reference labels only. Shared colour limits affect
display only. The reference is the CMEMS HR-OC product, not a pixel-level label map.

In [ ]:
from pathlib import Path
import sys
import os
import json
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from IPython.display import display

# Works when Jupyter starts in the repository root or in notebooks/.
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "aquacubeai").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Start Jupyter from the AquaCubeAI repository.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))


## 1. Configuration and paths

In [ ]:
from aquacubeai.config import data_directory, configuration_file
from aquacubeai.inference import run_reference_inference, save_inference_outputs
from aquacubeai.plotting import plot_mapping

# Optional: os.environ["AQUACUBEAI_DATA_DIR"] = r"/path/to/supplied/data"
DATA_ROOT = data_directory()
CONFIG_PATH = configuration_file("inference_20x20.json")
DEVICE = "cpu"  # Set to "cuda" to use a compatible GPU.
config = json.loads(CONFIG_PATH.read_text(encoding="utf-8"))
OUTPUT_DIR = ROOT / config["output_dir"]
print(f"Device: {DEVICE} | Data: {DATA_ROOT}")


## 2. Load and predict

Match the supplied CSV to patch filenames within 2.5 m, retain finite reference
targets in `[0, 140)` inside the GeoTIFF extent, and predict one dense map per
retained patch. The selected scene and coordinate system are configured in JSON.

In [ ]:
result = run_reference_inference(CONFIG_PATH, DATA_ROOT, device=DEVICE)
print(f"Scene: {result['scene']} | Retained patches: {len(result['reference_fnu']):,}")
print("Dense output shape:", result["predicted_maps_fnu"].shape)


## 3. Quantitative summary

Compare one **mean dense prediction per patch** against each corresponding
CMEMS reference target. This is `mean(f(pixel spectra))`; training/external-test
MLP metrics use `f(mean spectra)`. Bias is prediction minus reference.

In [ ]:
display(pd.DataFrame([result["metrics"]]))
maps = result["predicted_maps_fnu"]
display(
    pd.DataFrame(
        {
            "Percentile": [50, 95, 99, 99.9, 100],
            "Predicted FNU": np.percentile(maps, [50, 95, 99, 99.9, 100]),
        }
    )
)
print("Predicted pixel counts may repeat ground locations where patches overlap.")


## 4. Mapping figure and numerical outputs

The legacy visualization uses 300-m square overlays centered at reference
locations. It is a paper-style illustration, not a georeferenced full-scene
raster. Use the deployment workflow for full-scene reconstruction.

In [ ]:
save_inference_outputs(result, OUTPUT_DIR)
figure_path = OUTPUT_DIR / f"{result['scene']}_common_scale.png"
fig = plot_mapping(result, save_path=figure_path)
plt.show()
print("Saved figure and numerical outputs:", OUTPUT_DIR)
